# Topic 5 — Naive Bayes

> **Goal of this notebook:** understand the probabilistic foundation of Naive Bayes (Bayes' theorem + the "naive" independence assumption), meet its main variants, and apply it to both numeric data and text classification.

**Contents**
1. Concept & intuition
2. The mathematics (Bayes' theorem, the naive assumption)
3. The variants (Gaussian, Multinomial, Bernoulli)
4. The data: Wine (numeric) + a tiny spam corpus (text)
5. Gaussian Naive Bayes on Wine
6. Evaluation
7. Multinomial Naive Bayes for text
8. Gaussian NB from scratch
9. Pros, cons & when to use
10. Summary

## 1. Concept & Intuition

Naive Bayes is a **probabilistic classifier** built on **Bayes' theorem**. For each class it asks: *given these feature values, how probable is this class?* — then picks the most probable one.

The "naive" part: it assumes every feature is **conditionally independent** of the others given the class. This is almost never literally true, yet the classifier works remarkably well in practice (especially for text), is extremely fast, and needs little data.

## 2. The Mathematics

**Bayes' theorem:**
$$P(y \mid x_1,\dots,x_n) = \frac{P(y)\,P(x_1,\dots,x_n \mid y)}{P(x_1,\dots,x_n)}$$

The denominator is the same for every class, so we only compare numerators. The **naive independence** assumption factorises the likelihood:

$$P(x_1,\dots,x_n \mid y) = \prod_{i=1}^{n} P(x_i \mid y)$$

so the prediction is:

$$\hat{y} = \arg\max_{y}\; P(y) \prod_{i=1}^{n} P(x_i \mid y)$$

In practice we sum **log-probabilities** instead of multiplying many small numbers (to avoid underflow):

$$\hat{y} = \arg\max_{y}\; \log P(y) + \sum_{i=1}^{n} \log P(x_i \mid y)$$

## 3. The Variants

They differ in how $P(x_i \mid y)$ is modelled:

| Variant | Feature type | $P(x_i \mid y)$ model |
|---------|--------------|------------------------|
| **GaussianNB** | continuous | Normal distribution (mean, var per class) |
| **MultinomialNB** | counts | word/feature counts (great for text) |
| **BernoulliNB** | binary | presence/absence of features |

**Laplace smoothing** (`alpha`) adds a small count to every feature so an unseen value doesn't make the whole product zero.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

np.random.seed(42)
print('Libraries loaded.')

## 4. The Data

**Wine** (built-in): 178 wines, 13 chemical measurements, 3 cultivar classes — continuous features, perfect for GaussianNB. Plus a tiny inline **spam corpus** to demonstrate text classification with MultinomialNB.

In [ ]:
wine = load_wine()
X, y = wine.data, wine.target
print('Wine shape:', X.shape, '| classes:', np.bincount(y))
print('Features:', list(wine.feature_names[:5]), '...')

## 5. Gaussian Naive Bayes on Wine

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)
print('Test accuracy:', round(accuracy_score(y_test, y_pred), 4))

## 6. Evaluation

In [ ]:
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=wine.target_names))

## 7. Multinomial Naive Bayes for Text

Text is where Naive Bayes shines. We turn messages into word-count vectors, then classify spam vs ham.

In [ ]:
messages = [
    ('win a free prize now claim your reward', 'spam'),
    ('free money click this link to win cash', 'spam'),
    ('claim your free lottery prize urgent', 'spam'),
    ('cheap meds buy now limited offer', 'spam'),
    ('you won a free gift card click now', 'spam'),
    ('lets meet for lunch tomorrow', 'ham'),
    ('can you send me the report by friday', 'ham'),
    ('the meeting is rescheduled to monday', 'ham'),
    ('happy birthday hope you have a great day', 'ham'),
    ('please review the attached document', 'ham'),
]
texts = [m for m, _ in messages]
labels = [l for _, l in messages]

vec = CountVectorizer()
Xt = vec.fit_transform(texts)
nb_text = MultinomialNB().fit(Xt, labels)

tests = ['free prize click now to win', 'can we meet for lunch friday']
pred = nb_text.predict(vec.transform(tests))
for t, p in zip(tests, pred):
    print(f'[{p:>4}]  {t}')

## 8. Gaussian NB From Scratch

Fit a Normal distribution per feature per class, then apply Bayes' rule in log-space.

In [ ]:
class GaussianNBScratch:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.mean, self.var, self.prior = {}, {}, {}
        for c in self.classes:
            Xc = X[y == c]
            self.mean[c] = Xc.mean(axis=0)
            self.var[c] = Xc.var(axis=0) + 1e-9   # avoid divide-by-zero
            self.prior[c] = len(Xc) / len(X)
        return self

    def _log_gaussian(self, c, x):
        m, v = self.mean[c], self.var[c]
        return -0.5 * (np.log(2 * np.pi * v) + (x - m) ** 2 / v)

    def predict(self, X):
        out = []
        for x in X:
            scores = {c: np.log(self.prior[c]) + self._log_gaussian(c, x).sum()
                      for c in self.classes}
            out.append(max(scores, key=scores.get))
        return np.array(out)

In [ ]:
scratch = GaussianNBScratch().fit(X_train, y_train)
print('From-scratch accuracy:', round(accuracy_score(y_test, scratch.predict(X_test)), 4))
print('scikit-learn accuracy:', round(accuracy_score(y_test, y_pred), 4))

## 9. Pros, Cons & When to Use

**Pros**
- Extremely fast to train and predict; scales to huge feature counts.
- Works well with little training data.
- Excellent baseline for **text classification** (spam, sentiment, topics).

**Cons**
- The independence assumption is unrealistic → probability estimates are often poorly calibrated.
- Correlated features can hurt it.
- GaussianNB assumes each feature is roughly Normal per class.

**When to use**
- Text / high-dimensional sparse data.
- As a fast, strong baseline before heavier models.
- When training speed and simplicity matter.

## 10. Summary

- Naive Bayes combines **Bayes' theorem** with a **conditional-independence** assumption to pick the most probable class.
- Variants model $P(x\mid y)$ differently: **Gaussian** (continuous), **Multinomial** (counts/text), **Bernoulli** (binary).
- It classified Wine cultivars accurately and correctly flagged spam vs ham on raw text.
- Our from-scratch Gaussian NB matched scikit-learn, confirming the log-space Bayes computation.